In [ ]:
# =========================
# THESIS 4-SCENARIO PIPELINE
# Colab-ready | Google Drive output
# Dataset: All.Engineering.Papers_5000.csv
# Columns: doc_id, year, field, text_fa_raw, text_en_raw, pseudo_label
# =========================

!pip install -q pandas numpy scikit-learn matplotlib seaborn openpyxl xlsxwriter sentence-transformers transformers torch hazm

from google.colab import drive
drive.mount('/content/drive')

from __future__ import annotations
import os
import re
import json
import time
import math
import zipfile
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, PCA
from sklearn.preprocessing import normalize, MinMaxScaler, LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (
    silhouette_score,
    normalized_mutual_info_score,
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score
)

from sentence_transformers import SentenceTransformer

# =========================
# Persian/English text-cleaning resources (added)
# Uses hazm if available; otherwise falls back to a built-in stopword list.
# =========================
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
EN_STOP = set(ENGLISH_STOP_WORDS)

# Built-in fallback Persian stopword list (used only if hazm is unavailable)
_FA_STOP_FALLBACK = set("""
و در به از که این را با است برای آن یک تا های می ها هم خود نیز شده بر اما یا اگر
بود کرد باید چه چون هر همه دو شد دارد شود مورد بین حتی چند طور دیگر پس کنند نمی
بی روی همین بسیار بدون پیش وی ای آنها ایم اند کنیم کند داده مانند طی توسط همچنین
گرفته داشته باشد نسبت زیرا چنین آیا کدام کسی چرا کجا اینکه آنکه گردد گفت داشت
میشود میباشد بهعنوان نتایج روش مدل استفاده بهبود دهد دارند کرده شدن آنان خواهد
""".split())

_HAS_HAZM = False
try:
    from hazm import Normalizer as _HazmNormalizer, \
                     word_tokenize as _hazm_word_tokenize, stopwords_list as _hazm_stopwords_list
    _fa_normalizer = _HazmNormalizer()
    FA_STOP = set(_hazm_stopwords_list())
    _HAS_HAZM = True
    print("[preprocess] hazm loaded — using Normalizer + official stopwords (no stemming, fastest).")
except Exception as _e:
    FA_STOP = _FA_STOP_FALLBACK
    print("[preprocess] hazm NOT available — using regex normalizer + fallback stopword list.")

# Min token length to keep (drops most prepositions / single letters)
MIN_TOKEN_LEN = 3

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")


# =========================
# CONFIG
# =========================
@dataclass
class Config:
    input_csv: str = "/content/drive/MyDrive/deep-embedd/All.Engineering.Papers_5000.csv"
    out_root: str = "/content/drive/MyDrive/deep-embedd/thesis_outputs"

    doc_id_col: str = "doc_id"
    year_col: str = "year"
    field_col: str = "field"
    fa_col: str = "text_fa_raw"
    en_col: str = "text_en_raw"
    pseudo_col: str = "pseudo_label"

    random_state: int = 42

    min_df: int = 5
    max_df_ratio: float = 0.85
    max_features: int = 30000

    lda_k_grid: Tuple[int, ...] = (10, 15, 20, 25, 30)
    lda_max_iter: int = 20
    lda_learning_method: str = "batch"
    top_words_per_topic: int = 12

    topic_merge_cos_thr: float = 0.92
    embed_merge_cos_thr: float = 0.95

    rho_grid: Tuple[float, ...] = (0.70, 0.75, 0.80, 0.85, 0.90, 0.93)
    alpha: float = 1e-6
    beta: float = 1.0

    lambda_fa: float = 0.5
    en_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"
    fa_model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    batch_size_embed: int = 32

    fig_dpi: int = 300
    top_clusters_to_report: int = 10

CFG = Config()

SCENARIOS = {
    "S1_LDA_ENTM_SSFuzzyART": {"feature_type": "lda",  "use_entm": True},
    "S2_LDA_SSFuzzyART":      {"feature_type": "lda",  "use_entm": False},
    "S3_BERT_ENTM_SSFuzzyART":{"feature_type": "bert", "use_entm": True},
    "S4_BERT_SSFuzzyART":     {"feature_type": "bert", "use_entm": False},
}


# =========================
# HELPERS
# =========================
def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def save_json(obj, path):
    ensure_dir(Path(path).parent)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def normalize_fa(text):
    text = safe_text(text)
    if _HAS_HAZM:
        # hazm: standard normalization (chars, ZWNJ, diacritics)
        text = _fa_normalizer.normalize(text)
        # drop digits / latin / punctuation, keep Persian letters and spaces
        text = re.sub(r"[\u064B-\u0652\u0640]", "", text)
        text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        tokens = _hazm_word_tokenize(text)
        cleaned = [t for t in tokens
                   if t and t not in FA_STOP and len(t) >= MIN_TOKEN_LEN]   # no stemming (fastest)
        return " ".join(cleaned)
    # ---- fallback (no hazm) ----
    text = re.sub(r"[\u064B-\u0652\u0640]", "", text)            # diacritics + tatweel
    for a, b in {"ي":"ی","ك":"ک","أ":"ا","إ":"ا","آ":"ا","ٱ":"ا",
                 "ة":"ه","ؤ":"و","ئ":"ی"}.items():
        text = text.replace(a, b)                                  # unify Arabic chars
    text = text.replace("\u200c", " ")                             # ZWNJ
    text = re.sub(r"[0-9\u06F0-\u06F9]+", " ", text)              # digits
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)              # keep Persian only
    text = re.sub(r"\s+", " ", text).strip()
    toks = [t for t in text.split() if t not in FA_STOP and len(t) >= MIN_TOKEN_LEN]
    return " ".join(toks)

def normalize_en(text):
    text = safe_text(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)                          # drop digits + symbols
    text = re.sub(r"\s+", " ", text).strip()
    toks = [t for t in text.split() if t not in EN_STOP and len(t) >= MIN_TOKEN_LEN]
    return " ".join(toks)

def minmax_01(X):
    scaler = MinMaxScaler()
    return scaler.fit_transform(X)

def entropy_row(p):
    p = np.asarray(p, dtype=float)
    s = p.sum()
    if s <= 0:
        return 0.0
    p = p / s
    p = np.clip(p, 1e-12, 1.0)
    return float(-(p * np.log(p)).sum())

def topic_diversity(topic_words_lists):
    all_words = []
    for words in topic_words_lists:
        all_words.extend(words)
    if len(all_words) == 0:
        return np.nan
    return len(set(all_words)) / len(all_words)

def zip_all_outputs(src_dir: Path, zip_path: Path):
    ensure_dir(zip_path.parent)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in src_dir.rglob("*"):
            if p.is_file() and p != zip_path:
                zf.write(p, arcname=p.relative_to(src_dir))

def plot_bar(df, xcol, ycol, title, out_path, rotate=25):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(df[xcol].astype(str), df[ycol], alpha=0.9)
    ax.set_title(title)
    ax.set_xlabel(xcol)
    ax.set_ylabel(ycol)
    ax.tick_params(axis="x", rotation=rotate)
    fig.tight_layout()
    fig.savefig(out_path, dpi=CFG.fig_dpi)
    plt.close(fig)


# =========================
# DATA LOADING & PREPROCESS
# =========================
def load_dataset():
    df = pd.read_csv(CFG.input_csv, encoding="utf-8-sig")
    required = [CFG.fa_col, CFG.en_col, CFG.pseudo_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if CFG.doc_id_col not in df.columns:
        df.insert(0, CFG.doc_id_col, np.arange(1, len(df) + 1))

    return df

def preprocess_dataset(df: pd.DataFrame, out_dir: Path, force: bool = False):
    ensure_dir(out_dir)

    # ---- cache: if preprocessing already ran, just load it (skip the slow step) ----
    cache_path = out_dir / "preprocessed_dataset.csv"
    if cache_path.exists() and not force:
        print(f"[preprocess] cache found -> loading {cache_path} (set force=True to rebuild)")
        d = pd.read_csv(cache_path, encoding="utf-8-sig")
        for col in ["text_fa_clean", "text_en_clean", "text_joined"]:
            if col in d.columns:
                d[col] = d[col].fillna("").astype(str)
        d["has_fa"] = d["text_fa_clean"].str.len() > 0
        d["has_en"] = d["text_en_clean"].str.len() > 0
        return d

    print("[preprocess] building preprocessed dataset (this runs once)...")
    d = df.copy()
    d["text_fa_clean"] = d[CFG.fa_col].fillna("").astype(str).apply(normalize_fa)
    d["text_en_clean"] = d[CFG.en_col].fillna("").astype(str).apply(normalize_en)
    d["text_joined"] = (
        d["text_fa_clean"].fillna("") + " " + d["text_en_clean"].fillna("")
    ).str.strip()

    d["has_fa"] = d["text_fa_clean"].str.len() > 0
    d["has_en"] = d["text_en_clean"].str.len() > 0

    d.to_csv(out_dir / "preprocessed_dataset.csv", index=False, encoding="utf-8-sig")

    stats = {
        "n_docs": int(len(d)),
        "n_has_fa": int(d["has_fa"].sum()),
        "n_has_en": int(d["has_en"].sum()),
        "n_has_both": int((d["has_fa"] & d["has_en"]).sum()),
        "pseudo_label_nunique": int(d[CFG.pseudo_col].nunique())
    }
    save_json(stats, out_dir / "preprocess_stats.json")
    return d


# =========================
# LDA FEATURES
# =========================
def build_lda_features(df1: pd.DataFrame, out_dir: Path):
    ensure_dir(out_dir)

    vectorizer = CountVectorizer(
        min_df=CFG.min_df,
        max_df=CFG.max_df_ratio,
        max_features=CFG.max_features,
        stop_words=list(FA_STOP | EN_STOP),     # second safety layer
        token_pattern=r"(?u)\b\w{3,}\b"        # tokens with >= 3 chars
    )
    X_bow = vectorizer.fit_transform(df1["text_joined"].fillna(""))

    k_records = []
    best_score = None
    best_model = None
    best_k = None

    for k in CFG.lda_k_grid:
        lda = LatentDirichletAllocation(
            n_components=k,
            random_state=CFG.random_state,
            max_iter=CFG.lda_max_iter,
            learning_method=CFG.lda_learning_method
        )
        doc_topic = lda.fit_transform(X_bow)
        perp = lda.perplexity(X_bow)
        score = lda.score(X_bow)
        rec = {"k": k, "perplexity": float(perp), "log_likelihood": float(score)}
        k_records.append(rec)

        if best_score is None or score > best_score:
            best_score = score
            best_model = lda
            best_k = k

    k_df = pd.DataFrame(k_records)
    k_df.to_csv(out_dir / "lda_k_search.csv", index=False, encoding="utf-8-sig")

    lda_final = best_model
    doc_topic = lda_final.transform(X_bow)
    doc_topic = normalize(doc_topic, norm="l1")

    vocab = np.array(vectorizer.get_feature_names_out())
    topic_word = lda_final.components_ / lda_final.components_.sum(axis=1, keepdims=True)

    topics_rows = []
    topic_word_lists = []
    for t in range(topic_word.shape[0]):
        idx = np.argsort(topic_word[t])[::-1][:CFG.top_words_per_topic]
        words = vocab[idx].tolist()
        topic_word_lists.append(words)
        topics_rows.append({
            "topic_id": t,
            "top_words": " | ".join(words),
            "topic_entropy": entropy_row(topic_word[t])
        })
    topics_df = pd.DataFrame(topics_rows)
    topics_df.to_csv(out_dir / "lda_topics_top_words.csv", index=False, encoding="utf-8-sig")

    pd.DataFrame(doc_topic).to_csv(out_dir / "lda_doc_topic.csv", index=False, encoding="utf-8-sig")

    meta = {
        "feature_type": "lda",
        "best_k": int(best_k),
        "topic_diversity": float(topic_diversity(topic_word_lists)),
        "vocab_size": int(len(vocab))
    }
    save_json(meta, out_dir / "lda_meta.json")

    return {
        "X_features": doc_topic,
        "lda_model": lda_final,
        "vectorizer": vectorizer,
        "topic_word": topic_word,
        "topics_df": topics_df,
        "topic_word_lists": topic_word_lists,
        "meta": meta
    }


# =========================
# BERT / MULTILINGUAL FEATURES
# =========================
def build_bert_features(df1: pd.DataFrame, out_dir: Path):
    ensure_dir(out_dir)

    print("Loading embedding models...")
    model_en = SentenceTransformer(CFG.en_model_name)
    model_fa = SentenceTransformer(CFG.fa_model_name)

    fa_texts = df1["text_fa_clean"].fillna("").astype(str).tolist()
    en_texts = df1["text_en_clean"].fillna("").astype(str).tolist()

    print("Encoding Persian texts...")
    E_fa = model_fa.encode(
        fa_texts,
        batch_size=CFG.batch_size_embed,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    print("Encoding English texts...")
    E_en = model_en.encode(
        en_texts,
        batch_size=CFG.batch_size_embed,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    has_fa = df1["has_fa"].values.astype(bool)
    has_en = df1["has_en"].values.astype(bool)

    E = np.zeros_like(E_en, dtype=np.float32)

    for i in range(len(df1)):
        if has_fa[i] and has_en[i]:
            E[i] = CFG.lambda_fa * E_fa[i] + (1.0 - CFG.lambda_fa) * E_en[i]
        elif has_fa[i]:
            E[i] = E_fa[i]
        elif has_en[i]:
            E[i] = E_en[i]
        else:
            E[i] = np.zeros(E_en.shape[1], dtype=np.float32)

    E = normalize(E, norm="l2")
    X_embed = minmax_01(E)

    pd.DataFrame(X_embed).to_csv(out_dir / "bert_parsbert_embeddings.csv", index=False, encoding="utf-8-sig")

    meta = {
        "feature_type": "bert",
        "en_model": CFG.en_model_name,
        "fa_model": CFG.fa_model_name,
        "embed_dim": int(X_embed.shape[1]),
        "lambda_fa": float(CFG.lambda_fa)
    }
    save_json(meta, out_dir / "bert_meta.json")

    return {
        "X_features": X_embed,
        "meta": meta
    }


# =========================
# ENTM-LIKE STABILIZATION
# =========================
def merge_redundant_dimensions_by_similarity(X, threshold=0.95):
    X = np.asarray(X, dtype=float)
    XT = X.T
    sim = cosine_similarity(XT)

    n = sim.shape[0]
    visited = np.zeros(n, dtype=bool)
    groups = []

    for i in range(n):
        if visited[i]:
            continue
        grp = [i]
        visited[i] = True
        for j in range(i + 1, n):
            if not visited[j] and sim[i, j] >= threshold:
                grp.append(j)
                visited[j] = True
        groups.append(grp)

    X_new = np.zeros((X.shape[0], len(groups)), dtype=float)
    for g_idx, grp in enumerate(groups):
        X_new[:, g_idx] = X[:, grp].mean(axis=1)

    X_new = normalize(np.clip(X_new, 1e-12, None), norm="l1" if X_new.sum() > 0 else "l2")
    return X_new, groups, sim

def run_entm_general(X0, feature_type, out_dir: Path):
    ensure_dir(out_dir)

    thr = CFG.topic_merge_cos_thr if feature_type == "lda" else CFG.embed_merge_cos_thr
    X1, groups, sim = merge_redundant_dimensions_by_similarity(X0, threshold=thr)

    pd.DataFrame(X1).to_csv(out_dir / "entm_stabilized_features.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame({
        "group_id": list(range(len(groups))),
        "merged_dims": ["|".join(map(str, g)) for g in groups],
        "group_size": [len(g) for g in groups]
    }).to_csv(out_dir / "entm_groups.csv", index=False, encoding="utf-8-sig")

    plt.figure(figsize=(8, 6))
    sns.heatmap(sim, cmap="viridis")
    plt.title(f"Similarity matrix before ENTM-like merge ({feature_type})")
    plt.tight_layout()
    plt.savefig(out_dir / "entm_similarity_heatmap.png", dpi=CFG.fig_dpi)
    plt.close()

    meta = {
        "entm_applied": True,
        "feature_type": feature_type,
        "merge_threshold": float(thr),
        "dim_before": int(X0.shape[1]),
        "dim_after": int(X1.shape[1]),
        "n_groups": int(len(groups))
    }
    save_json(meta, out_dir / "entm_meta.json")
    return X1, meta


# =========================
# SIMPLE SSFuzzyART-LIKE CLUSTERER
# =========================
class SSFuzzyART:
    def __init__(self, rho=0.85, alpha=1e-6, beta=1.0):
        self.rho = rho
        self.alpha = alpha
        self.beta = beta
        self.prototypes = None
        self.counts = None

    def _choice(self, x, w):
        return np.sum(np.minimum(x, w)) / (self.alpha + np.sum(w))

    def _match(self, x, w):
        denom = max(np.sum(x), 1e-12)
        return np.sum(np.minimum(x, w)) / denom

    def fit_predict(self, X):
        X = np.asarray(X, dtype=float)
        labels = np.full(X.shape[0], -1, dtype=int)
        memberships = []

        prototypes = []
        counts = []

        for i, x in enumerate(X):
            if len(prototypes) == 0:
                prototypes.append(x.copy())
                counts.append(1)
                labels[i] = 0
                memberships.append([1.0])
                continue

            choices = np.array([self._choice(x, w) for w in prototypes])
            order = np.argsort(choices)[::-1]

            assigned = False
            soft = np.zeros(len(prototypes), dtype=float)

            for j in order:
                m = self._match(x, prototypes[j])
                soft[j] = m
                if m >= self.rho:
                    prototypes[j] = self.beta * np.minimum(x, prototypes[j]) + (1 - self.beta) * prototypes[j]
                    counts[j] += 1
                    labels[i] = j
                    assigned = True
                    break

            if not assigned:
                prototypes.append(x.copy())
                counts.append(1)
                labels[i] = len(prototypes) - 1
                soft = np.append(soft, 1.0)

            if soft.sum() <= 0:
                soft = np.zeros(len(prototypes))
                soft[labels[i]] = 1.0
            else:
                soft = soft / soft.sum()

            memberships.append(soft)

        self.prototypes = np.array(prototypes)
        self.counts = np.array(counts)
        max_len = max(len(m) for m in memberships)
        M = np.zeros((len(memberships), max_len))
        for i, m in enumerate(memberships):
            M[i, :len(m)] = m
        return labels, M


# =========================
# EVALUATION
# =========================
def evaluate_clustering(X, labels, pseudo_labels):
    out = {}
    n_clusters = len(np.unique(labels))
    out["n_clusters"] = int(n_clusters)

    if n_clusters >= 2 and len(X) > n_clusters:
        try:
            out["silhouette_cosine"] = float(silhouette_score(X, labels, metric="cosine"))
        except:
            out["silhouette_cosine"] = np.nan
        try:
            out["calinski_harabasz"] = float(calinski_harabasz_score(X, labels))
        except:
            out["calinski_harabasz"] = np.nan
        try:
            out["davies_bouldin"] = float(davies_bouldin_score(X, labels))
        except:
            out["davies_bouldin"] = np.nan
    else:
        out["silhouette_cosine"] = np.nan
        out["calinski_harabasz"] = np.nan
        out["davies_bouldin"] = np.nan

    le = LabelEncoder()
    y_true = le.fit_transform(pseudo_labels.astype(str))
    out["NMI_pseudo"] = float(normalized_mutual_info_score(y_true, labels))
    out["ARI_pseudo"] = float(adjusted_rand_score(y_true, labels))
    return out

def rho_sweep_and_select(X, pseudo_labels, out_dir: Path):
    ensure_dir(out_dir)

    rows = []
    best = None
    best_obj = -np.inf
    best_labels = None
    best_memberships = None

    for rho in CFG.rho_grid:
        model = SSFuzzyART(rho=rho, alpha=CFG.alpha, beta=CFG.beta)
        labels, memberships = model.fit_predict(X)
        metrics = evaluate_clustering(X, labels, pseudo_labels)
        metrics["rho"] = rho
        rows.append(metrics)

        sil = metrics["silhouette_cosine"]
        nmi = metrics["NMI_pseudo"]
        db = metrics["davies_bouldin"]

        sil2 = sil if not np.isnan(sil) else -1
        db2 = -db if not np.isnan(db) else -999
        obj = sil2 + nmi + 0.2 * db2

        if obj > best_obj:
            best_obj = obj
            best = metrics.copy()
            best_labels = labels.copy()
            best_memberships = memberships.copy()

    rho_df = pd.DataFrame(rows)
    rho_df.to_csv(out_dir / "rho_sweep_metrics.csv", index=False, encoding="utf-8-sig")

    plot_bar(rho_df, "rho", "silhouette_cosine", "Rho Sweep - Silhouette", out_dir / "rho_vs_silhouette.png", rotate=0)
    plot_bar(rho_df, "rho", "NMI_pseudo", "Rho Sweep - NMI", out_dir / "rho_vs_nmi.png", rotate=0)

    return best, best_labels, best_memberships, rho_df


# =========================
# REPORTING PER SCENARIO
# =========================
def save_cluster_outputs(df1, X_final, labels, memberships, summary, out_dir: Path):
    ensure_dir(out_dir)

    out_df = df1[[CFG.doc_id_col, CFG.year_col, CFG.field_col, CFG.pseudo_col, CFG.fa_col, CFG.en_col]].copy()
    out_df["cluster_id"] = labels
    out_df["max_membership"] = memberships.max(axis=1) if memberships.ndim == 2 else 1.0
    out_df.to_csv(out_dir / "final_doc_cluster_assignments.csv", index=False, encoding="utf-8-sig")

    np.save(out_dir / "final_doc_cluster_memberships.npy", memberships)

    cluster_sizes = (
        pd.Series(labels).value_counts().sort_index().reset_index()
    )
    cluster_sizes.columns = ["cluster_id", "size"]
    cluster_sizes.to_csv(out_dir / "cluster_size_distribution.csv", index=False, encoding="utf-8-sig")

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(cluster_sizes["cluster_id"].astype(str), cluster_sizes["size"])
    ax.set_title("Cluster Size Distribution")
    ax.set_xlabel("cluster_id")
    ax.set_ylabel("size")
    ax.tick_params(axis="x", rotation=25)
    fig.tight_layout()
    fig.savefig(out_dir / "cluster_size_distribution.png", dpi=CFG.fig_dpi)
    plt.close(fig)

    top_clusters = cluster_sizes.sort_values("size", ascending=False).head(CFG.top_clusters_to_report)
    top_clusters.to_csv(out_dir / "top_clusters_summary.csv", index=False, encoding="utf-8-sig")

    rep_docs = []
    for cid in top_clusters["cluster_id"].tolist():
        sub = out_df[out_df["cluster_id"] == cid].copy()
        if len(sub) == 0:
            continue
        rep = sub.iloc[0]
        rep_docs.append({
            "cluster_id": int(cid),
            "doc_id": rep[CFG.doc_id_col],
            "pseudo_label": rep[CFG.pseudo_col],
            "field": rep[CFG.field_col]
        })
    pd.DataFrame(rep_docs).to_csv(out_dir / "representative_docs_top_clusters.csv", index=False, encoding="utf-8-sig")

    if X_final.shape[1] >= 2:
        try:
            pca = PCA(n_components=2, random_state=CFG.random_state)
            Z = pca.fit_transform(X_final)
            fig, ax = plt.subplots(figsize=(7, 6))
            sc = ax.scatter(Z[:, 0], Z[:, 1], c=labels, s=18, cmap="tab20", alpha=0.75)
            ax.set_title("PCA scatter of final features by cluster")
            ax.set_xlabel("PC1")
            ax.set_ylabel("PC2")
            fig.tight_layout()
            fig.savefig(out_dir / "pca_clusters_scatter.png", dpi=CFG.fig_dpi)
            plt.close(fig)
        except:
            pass

    save_json(summary, out_dir / "scenario_summary.json")
    pd.DataFrame([summary]).to_csv(out_dir / "scenario_summary.csv", index=False, encoding="utf-8-sig")


# =========================
# RUN ONE SCENARIO
# =========================
def run_one_scenario(df1: pd.DataFrame, scenario_name: str, scenario_cfg: dict, out_root: Path):
    sc_dir = out_root / scenario_name
    ensure_dir(sc_dir)

    t0 = time.time()

    if scenario_cfg["feature_type"] == "lda":
        feat = build_lda_features(df1, sc_dir / "stage2_features")
        X0 = feat["X_features"]
        extra_metrics = {
            "topic_diversity": feat["meta"]["topic_diversity"],
            "best_k_topics": feat["meta"]["best_k"]
        }
    else:
        feat = build_bert_features(df1, sc_dir / "stage2_features")
        X0 = feat["X_features"]
        extra_metrics = {
            "topic_diversity": np.nan,
            "best_k_topics": np.nan
        }

    if scenario_cfg["use_entm"]:
        X_final, entm_meta = run_entm_general(X0, scenario_cfg["feature_type"], sc_dir / "stage3_entm")
    else:
        X_final = X0.copy()
        entm_meta = {"entm_applied": False, "dim_before": int(X0.shape[1]), "dim_after": int(X0.shape[1])}
        save_json(entm_meta, sc_dir / "stage3_entm" / "entm_meta.json")

    best_metrics, best_labels, best_memberships, rho_df = rho_sweep_and_select(
        X_final, df1[CFG.pseudo_col], sc_dir / "stage4_5_cluster_eval"
    )

    runtime_total = time.time() - t0

    summary = {
        "scenario": scenario_name,
        "feature_type": scenario_cfg["feature_type"],
        "use_entm": scenario_cfg["use_entm"],
        "n_docs": int(X_final.shape[0]),
        "feature_dim_final": int(X_final.shape[1]),
        "runtime_total_sec": round(runtime_total, 3),
        **extra_metrics,
        **best_metrics
    }

    save_cluster_outputs(
        df1=df1,
        X_final=X_final,
        labels=best_labels,
        memberships=best_memberships,
        summary=summary,
        out_dir=sc_dir
    )

    return summary


# =========================
# COMPARISON
# =========================
def compare_scenarios(all_results: List[dict], out_dir: Path):
    ensure_dir(out_dir)

    comp_df = pd.DataFrame(all_results)
    comp_df.to_csv(out_dir / "scenarios_metrics_comparison.csv", index=False, encoding="utf-8-sig")

    rank_df = comp_df.copy()

    high_better = ["silhouette_cosine", "calinski_harabasz", "NMI_pseudo", "ARI_pseudo", "topic_diversity"]
    low_better = ["davies_bouldin", "runtime_total_sec"]

    for c in high_better:
        if c in rank_df.columns:
            x = rank_df[c].astype(float)
            rank_df[c + "_norm"] = (x - x.min()) / (x.max() - x.min() + 1e-12)

    for c in low_better:
        if c in rank_df.columns:
            x = rank_df[c].astype(float)
            rank_df[c + "_norm"] = (x.max() - x) / (x.max() - x.min() + 1e-12)

    norm_cols = [c for c in rank_df.columns if c.endswith("_norm")]
    rank_df["final_score"] = rank_df[norm_cols].mean(axis=1)
    rank_df = rank_df.sort_values("final_score", ascending=False)

    rank_df.to_csv(out_dir / "scenarios_ranked.csv", index=False, encoding="utf-8-sig")

    with pd.ExcelWriter(out_dir / "scenarios_comparison.xlsx", engine="xlsxwriter") as writer:
        comp_df.to_excel(writer, sheet_name="metrics", index=False)
        rank_df.to_excel(writer, sheet_name="ranking", index=False)

    figs_dir = out_dir / "figures"
    ensure_dir(figs_dir)

    metric_list = [
        "silhouette_cosine",
        "calinski_harabasz",
        "davies_bouldin",
        "NMI_pseudo",
        "ARI_pseudo",
        "n_clusters",
        "runtime_total_sec",
        "feature_dim_final"
    ]
    for m in metric_list:
        if m in comp_df.columns:
            plot_bar(comp_df, "scenario", m, f"Scenario Comparison - {m}", figs_dir / f"compare_{m}.png")

    return comp_df, rank_df


# =========================
# MAIN
# =========================
def main():
    out_root = Path(CFG.out_root)
    ensure_dir(out_root)

    print("Loading dataset...")
    df = load_dataset()

    print("Preprocessing...")
    df1 = preprocess_dataset(df, out_root / "common_preprocess")

    all_results = []

    for scenario_name, scenario_cfg in SCENARIOS.items():
        print(f"\nRunning {scenario_name} ...")
        result = run_one_scenario(df1, scenario_name, scenario_cfg, out_root)
        all_results.append(result)

    print("\nComparing scenarios...")
    comp_df, rank_df = compare_scenarios(all_results, out_root / "comparison")

    print("\nCreating final ZIP archive...")
    zip_all_outputs(out_root, out_root / "archives" / "all_outputs.zip")

    print("\nDONE.")
    print(f"All outputs saved to: {CFG.out_root}")

main()
